# Reranking Fundamentals

**Module:** 03 — Reranking

Reranking is the precision stage after fast retrieval. Learn the two-stage pattern, when it pays off, and how to reason about candidate depth, cost, and failure modes.


## How to Use This Notebook

Read each section as a mini-lesson, run every code cell, then change inputs to stress-test your intuition. API examples use placeholders such as `YOUR_API_KEY` or `os.getenv(...)` — never hard-code secrets.

Each major topic includes: definition, why it matters, how it works, intuition, pitfalls, when-to-use guidance, practical demos, and a short exercise.


## Learning Objectives

By the end of this notebook, you will be able to:

- Define reranking and contrast it with first-stage retrieval
- Explain the recall→precision handoff in dual-stage search
- Sketch a minimal retrieve → rerank → pack loop
- Identify when reranking helps vs when it wastes latency
- Name production pitfalls (depth, calibration, stale models)


## What is Reranking?

**Definition.** **Reranking** takes a shortlist of candidates from a cheap first-stage retriever (BM25, bi-encoder ANN, hybrid) and reorders them with a stronger, usually slower scorer—most often a **cross-encoder**—so the top-k is more precise.

**Why it matters.** LLM context windows and user attention are tiny. Putting the best evidence first raises answer quality more than retrieving 10× more weak hits.

**How it works.** Retrieve top-N (often 20–200) → score each (query, doc) pair → sort by score → keep top-k for the prompt or UI.

**Intuition.** A fast librarian pulls a cart of books; an expert skims the cart and stacks the best ones on top before you read.

**Common pitfalls.**
- Reranking an empty or tiny candidate set (nothing to reorder)
- Treating first-stage scores and rerank scores as interchangeable
- Calling any second sort 'reranking' without a stronger relevance model
- Ignoring that cross-encoders do not scale to the full corpus

**When to use.** Use when first-stage recall is decent but precision@k for the LLM/UI is weak.

```mermaid
flowchart LR
  Q[Query] --> R[First-stage retrieve N]
  R --> CE[Rerank scorer]
  CE --> K[Top-k]
  K --> P[Pack / UI / LLM]
```

| Stage | Goal | Typical tech | Cost |
|-------|------|--------------|------|
| First | High recall | BM25, bi-encoder, hybrid | Cheap / massively parallel |
| Second | High precision@k | Cross-encoder, LLM judge | Costly per pair |
| Pack | Fit context | Truncate, diversify, cite | Cheap |


In [ ]:
# Demo 1 — two-stage contract
from dataclasses import dataclass

@dataclass
class Hit:
    id: str
    text: str
    stage1: float
    stage2: float | None = None

candidates = [
    Hit("d1", "Cats are mammals.", 0.71),
    Hit("d2", "The cat sat on the mat in the living room.", 0.70),
    Hit("d3", "Feline vaccination schedule.", 0.69),
]
query = "where did the cat sit?"

def toy_ce(q: str, doc: str) -> float:
    qset, dset = set(q.lower().split()), set(doc.lower().split())
    overlap = len(qset & dset) / (len(qset) + 1e-9)
    return 0.7 * overlap + 0.3 * (1.0 / (1 + abs(len(doc.split()) - 8)))

for h in candidates:
    h.stage2 = toy_ce(query, h.text)
ranked = sorted(candidates, key=lambda h: h.stage2 or 0, reverse=True)
for h in ranked:
    print(f"{h.id} s1={h.stage1:.2f} s2={h.stage2:.3f} | {h.text}")


In [ ]:
# Demo 2 — candidate depth vs top-k for the LLM
def pipeline_shape(n_retrieve: int, k_context: int) -> dict:
    assert n_retrieve >= k_context
    return {
        "retrieve_N": n_retrieve,
        "rerank_pairs": n_retrieve,
        "pack_top_k": k_context,
        "discarded_after_rerank": n_retrieve - k_context,
    }

for n, k in [(20, 5), (50, 8), (100, 8), (200, 12)]:
    print(pipeline_shape(n, k))


In [ ]:
# Demo 3 — API-shaped rerank request (placeholder)
import json
YOUR_API_KEY = "YOUR_API_KEY"
req = {
    "model": "rerank-english-v3.0",
    "query": "refund window",
    "documents": [
        "Returns accepted within 30 days.",
        "Shipping takes 3–5 business days.",
        "Refunds within 60 days of purchase.",
    ],
    "top_n": 3,
}
resp = {
    "results": [
        {"index": 2, "relevance_score": 0.92},
        {"index": 0, "relevance_score": 0.71},
        {"index": 1, "relevance_score": 0.18},
    ]
}
print(json.dumps({"request": req, "response": resp}, indent=2))
print("Authorization: Bearer", YOUR_API_KEY[:8] + "...")


### Try it yourself — What is Reranking?

1. Explain reranking in two sentences to a backend engineer.
2. Pick N and k for a support FAQ bot with a 4k-token context budget.
3. List two signals a cross-encoder can use that ANN alone often misses.


## Why It Matters

**Definition.** Dual-stage retrieval separates **recall** (get the right docs into the pool) from **precision** (order the pool so the best evidence is used).

**Why it matters.** RAG failures are often 'right doc in top-50, wrong doc in top-5.' Reranking targets that gap without scanning the whole corpus with a cross-encoder.

**How it works.** Measure offline (nDCG@k, MRR) and online (deflection, thumbs, groundedness). Compare baseline retrieve-only vs retrieve+rerank under a latency SLA.

**Intuition.** A wider net, then a finer sieve—never try to sieve the ocean.

**Common pitfalls.**
- Optimizing rerank while first-stage recall@N is already broken
- Burning p95 latency on N=200 when N=40 was enough
- No A/B or offline harness—shipping on vibes
- Assuming higher CE scores mean factual truth

**When to use.** Prioritize when context slots are scarce, corpus is noisy, or paraphrase/ID mix hurts ANN.

### First stage vs second stage

| Concern | First stage | Rerank stage |
|---------|-------------|--------------|
| Scale | Millions of docs | Tens–hundreds of docs |
| Objective | Recall@N | Precision@k / nDCG@k |
| Failure | Miss gold entirely | Gold present but poorly ordered |


In [ ]:
# Demo 1 — synthetic recall@N vs precision@k
import random
random.seed(0)

def simulate(n_trials=200, gold_rank_mean=15, k=5):
    hit_at_k_no = 0
    hit_at_k_yes = 0
    for _ in range(n_trials):
        gold = max(1, int(random.gauss(gold_rank_mean, 5)))
        hit_at_k_no += int(gold <= k)
        in_pool = gold <= 50
        after = random.randint(1, 3) if in_pool and random.random() < 0.7 else gold
        hit_at_k_yes += int(after <= k)
    return hit_at_k_no/n_trials, hit_at_k_yes/n_trials

p0, p1 = simulate()
print(f"P(gold in top-5) retrieve-only≈{p0:.2f}  +rerank≈{p1:.2f}")


In [ ]:
# Demo 2 — when reranking helps most
scenarios = [
    ("paraphrase FAQ", True, "CE reads full pair semantics"),
    ("exact SKU / error code", False, "lexical first-stage often enough"),
    ("noisy web chunks", True, "CE demotes boilerplate"),
    ("N=5 already perfect", False, "no headroom"),
]
for name, helps, why in scenarios:
    print(f"{'HELP' if helps else 'SKIP':4s} | {name:22s} | {why}")


In [ ]:
# Demo 3 — cost sketch per 1k queries
def cost_per_1k(n_candidates: int, price_per_1k_pairs: float = 1.0) -> float:
    pairs = 1000 * n_candidates
    return pairs / 1000 * price_per_1k_pairs

for n in [20, 50, 100]:
    print(f"N={n:3d} → ${cost_per_1k(n):.1f} / 1k queries (placeholder pricing)")


### Try it yourself — Why It Matters

1. Name a query class where you would skip reranking.
2. If gold is usually around rank 30, what N would you start with and why?


## The retrieve → rerank → pack loop

**Definition.** Production systems treat reranking as one stage in a logged pipeline: retrieve, optional filters, rerank, diversity/pack, generate.

**Why it matters.** Debugging requires knowing which stage failed—missed recall vs bad order vs bad packing.

**How it works.** Emit traces: candidate IDs, stage1 scores, stage2 scores, final packed IDs, latency.

**Intuition.** If you cannot replay why doc A beat doc B, you cannot improve the system.

**Common pitfalls.**
- Silent truncation before rerank (different N in prod vs eval)
- Packing by stage1 score after computing stage2
- No fallback when the rerank API times out

**When to use.** Always—even a toy project benefits from explicit stage boundaries.

```mermaid
flowchart TB
  Q[Query] --> F[Filters / ACL]
  F --> R[Retrieve N]
  R --> RR[Rerank]
  RR -->|timeout| FB[Fallback: stage1 order]
  RR --> P[Pack top-k]
  FB --> P
  P --> L[LLM]
```


In [ ]:
# Demo 1 — staged pipeline object
from dataclasses import dataclass, field

@dataclass
class RagTrace:
    query: str
    retrieved: list[str] = field(default_factory=list)
    reranked: list[str] = field(default_factory=list)
    packed: list[str] = field(default_factory=list)
    latencies_ms: dict = field(default_factory=dict)

def run(q: str) -> RagTrace:
    t = RagTrace(query=q)
    t.retrieved = ["d3", "d1", "d2", "d9", "d4"]
    t.latencies_ms["retrieve"] = 35
    t.reranked = ["d1", "d2", "d3", "d4", "d9"]
    t.latencies_ms["rerank"] = 120
    t.packed = t.reranked[:3]
    t.latencies_ms["pack"] = 2
    return t

print(run("refund window"))


In [ ]:
# Demo 2 — timeout fallback
def with_fallback(rerank_fn, docs, timeout_ok: bool):
    if not timeout_ok:
        return docs
    return rerank_fn(docs)

docs = ["a", "b", "c"]
print("ok", with_fallback(lambda d: list(reversed(d)), docs, True))
print("timeout", with_fallback(lambda d: list(reversed(d)), docs, False))


In [ ]:
# Demo 3 — assert pack uses rerank order
reranked = ["d1", "d2", "d3"]
packed = reranked[:2]
assert packed == ["d1", "d2"]
print("pack order ok")


### Try it yourself — The retrieve → rerank → pack loop

1. Add one log field you would need to debug a bad answer.
2. Define a fallback policy for rerank p95 > 300ms.


## Glossary

- **first-stage retrieval**: Cheap broad candidate generation
- **cross-encoder**: Joint query-document Transformer scorer
- **candidate depth N**: How many hits enter the reranker
- **top-k**: How many hits enter the prompt/UI


### Workshop drill — Reranking Fundamentals (1)

Restate each major section heading as one exam-ready sentence.


In [ ]:
# Workshop drill 1 — Reranking Fundamentals
headings = ['What is Reranking?', 'Why It Matters', 'The retrieve → rerank → pack loop']
for h in headings:
    print('-', h, '→', '...')


### Workshop drill — Reranking Fundamentals (2)

Sketch a latency budget: first-stage ms + rerank (N candidates × cost) + LLM.


In [ ]:
# Workshop drill 2 — Reranking Fundamentals
first_ms, per_pair_ms, n, llm_ms = 40, 3, 50, 800
print('total_ms', first_ms + n*per_pair_ms + llm_ms)
print('rerank_share', round(n*per_pair_ms/(first_ms+n*per_pair_ms+llm_ms), 3))


### Workshop drill — Reranking Fundamentals (3)

Design an offline metric slice: 5 queries with graded relevance labels.


In [ ]:
# Workshop drill 3 — Reranking Fundamentals
eval_set = [{'q':'...','docs':{'d1':2,'d2':1,'d3':0}}]
print('n_queries', len(eval_set))
print('TODO: fill real labels')


### Workshop drill — Reranking Fundamentals (4)

Write a go/no-go checklist for shipping a reranker in RAG.


In [ ]:
# Workshop drill 4 — Reranking Fundamentals
for c in ['latency_p95','nDCG@10','cost/1k','cache_hit','fallback']:
    print(f'[ ] {c}')


### Workshop drill — Reranking Fundamentals (5)

Compare bi-encoder vs cross-encoder in a small table (fill TODOs).


In [ ]:
# Workshop drill 5 — Reranking Fundamentals
print('| axis | bi | cross |')
print('|------|----|-------|')
print('| latency | TODO | TODO |')
print('| precision | TODO | TODO |')


## Summary & Key Takeaways

- Reranking reorders a shortlist; it does not replace corpus-scale retrieval
- First stage buys recall; rerank buys precision@k for scarce context
- Choose N from where gold usually appears; choose k from the prompt budget
- Always log both scores and keep a timeout fallback

### Practice

Implement the toy CE on 10 FAQ chunks and compare top-3 before/after.


## Self-Check

1. Can you explain the main idea of each section in one sentence?
2. Which technique would you use first in production, and why?
3. What failure mode should you monitor after shipping?
4. What metric would tell you the system got worse?


In [ ]:
checklist = [
    "I can restate the learning objectives",
    "I ran/adapted at least two code examples",
    "I know which env vars/keys this topic needs",
    "I noted one risk (cost, safety, latency, or quality)",
    "I can name one pitfall and its mitigation",
]
for i, item in enumerate(checklist, 1):
    print(f"{i}. [ ] {item}")
